In [ ]:
"""
Explore the Indian Crops Disease Dataset
Run this in Jupyter notebook or as a Python script
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set paths
BASE_PATH = '../data/raw/Final_Dataset'
WHEAT_PATH = os.path.join(BASE_PATH, 'Wheat')
RICE_PATH = os.path.join(BASE_PATH, 'Rice')

def explore_dataset(crop_path, crop_name):
    """Explore the dataset structure"""
    print(f"\n{'='*50}")
    print(f"Exploring {crop_name} Dataset")
    print('='*50)
    
    # Get class names
    classes = [d for d in os.listdir(crop_path) 
               if os.path.isdir(os.path.join(crop_path, d))]
    print(f"\nNumber of classes: {len(classes)}")
    print(f"Classes: {classes}")
    
    # Count images per class
    image_counts = {}
    for class_name in classes:
        class_path = os.path.join(crop_path, class_name)
        images = [f for f in os.listdir(class_path) 
                 if f.endswith(('.jpg', '.jpeg', '.png'))]
        image_counts[class_name] = len(images)
        print(f"  {class_name}: {len(images)} images")
    
    # Display sample images
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.ravel()
    
    for idx, class_name in enumerate(classes[:8]):  # Show first 8 classes
        class_path = os.path.join(crop_path, class_name)
        images = [f for f in os.listdir(class_path) 
                 if f.endswith(('.jpg', '.jpeg', '.png'))]
        if images:
            img_path = os.path.join(class_path, images[0])
            img = Image.open(img_path)
            axes[idx].imshow(img)
            axes[idx].set_title(f"{class_name}\n{img.size}")
            axes[idx].axis('off')
    
    plt.suptitle(f"Sample Images from {crop_name} Dataset")
    plt.tight_layout()
    plt.show()
    
    return image_counts

# Explore both datasets
wheat_counts = explore_dataset(WHEAT_PATH, "Wheat")
rice_counts = explore_dataset(RICE_PATH, "Rice")

# Plot distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Wheat distribution
ax1.bar(wheat_counts.keys(), wheat_counts.values())
ax1.set_title('Wheat Disease Classes Distribution')
ax1.set_xlabel('Disease Class')
ax1.set_ylabel('Number of Images')
ax1.tick_params(axis='x', rotation=45)

# Rice distribution
ax2.bar(rice_counts.keys(), rice_counts.values())
ax2.set_title('Rice Disease Classes Distribution')
ax2.set_xlabel('Disease Class')
ax2.set_ylabel('Number of Images')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Create data generators for training
def create_data_generators():
    """Create training and validation data generators"""
    
    # Data augmentation for training
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        validation_split=0.2  # 80% train, 20% validation
    )
    
    # Only rescaling for validation
    val_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.2
    )
    
    # Create generators for wheat
    wheat_train = train_datagen.flow_from_directory(
        WHEAT_PATH,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        subset='training'
    )
    
    wheat_val = val_datagen.flow_from_directory(
        WHEAT_PATH,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        subset='validation'
    )
    
    # Create generators for rice
    rice_train = train_datagen.flow_from_directory(
        RICE_PATH,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        subset='training'
    )
    
    rice_val = val_datagen.flow_from_directory(
        RICE_PATH,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        subset='validation'
    )
    
    print("\n✅ Data generators created successfully!")
    print(f"Wheat classes: {wheat_train.class_indices}")
    print(f"Rice classes: {rice_train.class_indices}")
    
    return (wheat_train, wheat_val), (rice_train, rice_val)

# Create generators
(wheat_train, wheat_val), (rice_train, rice_val) = create_data_generators()